# Review training results


In [1]:
from pathlib import Path
import sys

import pandas as pd
import torch
from IPython.display import display

# Works when launched from Research, Pipeline2, or model_notebooks.
PROJECT_DIR = next(
    folder if (folder / "main.py").is_file() else folder / "Pipeline2"
    for folder in [Path.cwd(), *Path.cwd().parents]
    if (folder / "models" / "main_trainer.py").is_file()
    or (folder / "Pipeline2" / "models" / "main_trainer.py").is_file()
)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from utils.data_helper import TARGETS

# Change this to review a different run.
RUN_DIR = PROJECT_DIR / "runs" / "version_0"
CHECKPOINT_PATH = RUN_DIR / "checkpoints" / "best.ckpt"
metrics = pd.read_csv(RUN_DIR / "metrics.csv")

# Load only your own trusted checkpoint; keep it on CPU.
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
best_epoch = int(checkpoint["epoch"])
best_step = int(checkpoint["global_step"])
del checkpoint  # Release the model and optimizer tensors from memory.
print("Run:", RUN_DIR)
print("Best checkpoint:", CHECKPOINT_PATH)
print("Checkpoint epoch index:", best_epoch)
print("Completed epochs at checkpoint:", best_epoch + 1)
print("Optimizer steps at checkpoint:", best_step)

Run: /home/rittick-dutta/Desktop/Research/Pipeline2/runs/version_0
Best checkpoint: /home/rittick-dutta/Desktop/Research/Pipeline2/runs/version_0/checkpoints/best.ckpt
Checkpoint epoch index: 2
Completed epochs at checkpoint: 3
Optimizer steps at checkpoint: 3750


## Validation scores after each completed epoch
Lightning writes training and validation values to separate CSV rows. Select the rows containing final-model validation scores.

In [2]:
ap_column = "val_PRAUC_avg_over_dxs/final"
roc_column = "val_AUROC_avg_over_dxs/final"
validation = (
    metrics.dropna(subset=[ap_column])
    .sort_values(["epoch", "step"])
    .drop_duplicates(subset="epoch", keep="last")
    .copy()
)
validation["epoch"] = validation["epoch"].astype(int)

history = validation[["epoch", ap_column, roc_column]].rename(columns={
    "epoch": "epoch_index",
    ap_column: "validation_macro_AP",
    roc_column: "validation_macro_AUROC",
})
history.insert(1, "completed_epochs", history["epoch_index"] + 1)
history["saved_best_checkpoint"] = history["epoch_index"].eq(best_epoch)
display(history.reset_index(drop=True).round(4))

# Read the metrics belonging to the checkpoint, rather than the latest epoch.
best_row = validation.set_index("epoch").loc[best_epoch]
print(f"Best checkpoint: AP={best_row[ap_column]:.4f}, AUROC={best_row[roc_column]:.4f}")

,epoch_index,completed_epochs,validation_macro_AP,validation_macro_AUROC,saved_best_checkpoint
0,0,1,0.8298,0.6904,False
1,1,2,0.8418,0.7048,False
2,2,3,0.8580,0.7290,True
3,3,4,0.8490,0.6870,False
4,4,5,0.8494,0.6913,False
5,5,6,0.8539,0.7193,False


Best checkpoint: AP=0.8580, AUROC=0.7290


## Best checkpoint: final model and individual modality heads
The CXR head is evaluated only on admissions whose CXR is available. The final model includes admissions with and without CXR.

In [3]:
head_scores = pd.DataFrame([
    {
        "prediction_head": head,
        "validation_macro_AP": best_row[f"val_PRAUC_avg_over_dxs/{head}"],
        "validation_macro_AUROC": best_row[f"val_AUROC_avg_over_dxs/{head}"],
    }
    for head in ["final", "ecg", "labs", "cxr"]
])
display(head_scores.round(4))

,prediction_head,validation_macro_AP,validation_macro_AUROC
0,final,0.8580,0.7290
1,ecg,0.8017,0.5613
2,labs,0.8193,0.6205
3,cxr,0.8594,0.7160


## Best checkpoint: all 14 findings
These are the final fused model’s per-finding validation scores. No Finding can have AP=1 while AUROC is undefined when its observed labels contain only positives.

In [4]:
finding_scores = pd.DataFrame({
    "finding": TARGETS,
    "validation_AP": [best_row[f"val_PRAUC_per_dx_final/{name}"] for name in TARGETS],
    "validation_AUROC": [best_row[f"val_AUROC_per_dx_final/{name}"] for name in TARGETS],
})
display(finding_scores.round(4))
print("Findings included in macro AP:", finding_scores["validation_AP"].notna().sum())
print("Findings included in macro AUROC:", finding_scores["validation_AUROC"].notna().sum())

,finding,validation_AP,validation_AUROC
0,Atelectasis,0.9996,0.9813
1,Cardiomegaly,0.8912,0.7494
2,Consolidation,0.8706,0.7167
3,Edema,0.7294,0.7653
4,Enlarged Cardiomediastinum,0.8169,0.6500
5,Fracture,0.9787,0.7143
6,Lung Lesion,1.0000,NaN
7,Lung Opacity,0.9801,0.7343
8,No Finding,1.0000,NaN
9,Pleural Effusion,0.8515,0.7774


Findings included in macro AP: 14
Findings included in macro AUROC: 11
